In [1]:
%pip install xgboost optuna


   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   --------------------------

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import kagglehub

# General settings. Do not change TEST_SIZE
RANDOM_SEED = 42
TEST_SIZE = 0.3

# Load dataset (from kagglehub)
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(f"{path}/creditcard.csv")
data['Class'] = data['Class'].astype(int)

# Prepare data
data = data.drop(['Time'], axis=1)
data['Amount'] = StandardScaler().fit_transform(data['Amount'].values.reshape(-1, 1))

fraud = data[data['Class'] == 1]
nonfraud = data[data['Class'] == 0]
print(f'Fraudulent: {len(fraud)}, non-fraudulent: {len(nonfraud)}')
print(f'The positive class (frauds) percentage: {len(fraud)}/{len(fraud) + len(nonfraud)} ({len(fraud)/(len(fraud) + len(nonfraud))*100:.3f}%)')

X = np.asarray(data.iloc[:, ~data.columns.isin(['Class'])])
Y = np.asarray(data['Class'])

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=TEST_SIZE, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Calculate the ratio of fraud to non-fraud transactions
contamination = len(fraud) / len(nonfraud)
scale_pos_weight = len(nonfraud) / len(fraud)


c:\Users\mirol\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fraudulent:492, non-fraudulent:284315
the positive class (frauds) percentage: 492/284807 (0.173%)


In [ ]:
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# Step 1: Use IsolationForest for preliminary anomaly detection to generate a new feature
iso = IsolationForest(
    contamination=contamination,  # Proportion of anomalies in the data
    n_estimators=200,             # Number of trees
    max_samples='auto',           # Automatically choose the number of samples
    random_state=RANDOM_SEED
)

# Predict anomalies on the training set: returns +1 for normal, -1 for anomalies
iso_train_pred = iso.fit_predict(X_train)
# Predict anomalies on the test set
iso_test_pred = iso.predict(X_test)

# Convert IsolationForest output to 0/1 (1 represents anomaly)
iso_train_feature = (iso_train_pred == -1).astype(int).reshape(-1, 1)
iso_test_feature = (iso_test_pred == -1).astype(int).reshape(-1, 1)

# Concatenate the new anomaly feature with the original features
X_train_enhanced = np.hstack((X_train, iso_train_feature))
X_test_enhanced = np.hstack((X_test, iso_test_feature))


In [ ]:
# Step 2: Use XGBoost for supervised classification
xgb_model = XGBClassifier(
    n_estimators=210,         # Number of trees (weak learners); more trees may improve accuracy but increase computation cost
    learning_rate=0.17,       # Learning rate controls each tree’s contribution to the final prediction; smaller values improve generalization
    max_depth=10,             # Maximum tree depth controls model complexity and risk of overfitting
    subsample=1,              # Proportion of samples used for training each tree; helps reduce overfitting
    colsample_bytree=0.75,    # Proportion of features used for training each tree; helps prevent feature interaction overfitting
    gamma=0.5,                # Minimum loss reduction required to make a split; larger values make the algorithm more conservative
    scale_pos_weight=scale_pos_weight,  # Balances class weights to handle class imbalance, improving detection of minority (fraud) class
    min_child_weight=1,       # Minimum sum of instance weights in a child node; prevents learning patterns from small sample anomalies
    tree_method='hist',       # Use histogram-based algorithm for faster training; suitable for large datasets
    eval_metric='logloss',    # Evaluation metric; log loss evaluates the accuracy of predicted probabilities
    random_state=RANDOM_SEED  # Random seed for reproducibility
)

xgb_model.fit(X_train_enhanced, y_train.ravel())

# Make predictions and evaluate the model
y_pred = xgb_model.predict(X_test_enhanced)


In [ ]:
# Define evaluation function to display common classification metrics
def evaluation(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{model_name} Evaluation:')
    print('===' * 15)
    print('         Accuracy:', accuracy)
    print('  Precision Score:', precision)
    print('     Recall Score:', recall)
    print('         F1 Score:', f1)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# Evaluate the model on the test set
evaluation(y_test, y_pred, model_name="XGBClassifier")



XGBClassifier Evaluation:
         Accuracy: 0.9996488887328394
  Precision Score: 0.9206349206349206
     Recall Score: 0.8529411764705882
         F1 Score: 0.8854961832061069

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85307
           1       0.92      0.85      0.89       136

    accuracy                           1.00     85443
   macro avg       0.96      0.93      0.94     85443
weighted avg       1.00      1.00      1.00     85443

